In [1]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.tail()

,id,split,text,region,Tags
9018,77,valid,"['Fói’a', 'oficiale', 'din', 'Belgradu', 'publ...",Transylvania,"[0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, ..."
9019,926,valid,"['Mai', 'multu', 'se', 'intereséza', 'lumea', ...",Transylvania,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9020,8366,valid,"['Doar', 'pentru', 'atât', ',', 'păcat', 'ar',...",Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9021,5990,valid,"[')', 'România', 'perde', 'dar', 'o', 'provinc...",Wallachia,"[0, 0, 0, 0, 0, 0, 0]"
9022,8452,valid,"['Uneori', 'un', 'cuvînt', 'aspru', 'sau', 'de...",Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [2]:
# acum am inteles ca fiecare token e clasificat gen si de asta variaza lungimile listelor de taguri

good_train = df_train[df_train.split == "train"]

i=1
print(df_train.text[i])
print(df_train.Tags[i])

1 ) Asupra cestiunii , avem în romaneste oarecare noţiuni în doi lucrări : a ) Critica apriorismului si empirismului , de Petru P. NeguJescu , p. 125 sqq .
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:
#mai intai refac datasetul, repar alea din test sa fie la fel toate si scot splitul dat de ei

def fix_text(tokens):
    tokens = tokens.replace('[', '')
    tokens = tokens.replace(']', '')
    tokens = tokens.replace(r"'", '')
    tokens = tokens.replace(r",", '')
    return tokens

cond = df_train.split=='valid'
df_train.loc[cond, "text"] = df_train.loc[cond, "text"].apply(fix_text)
df_train.tail()

,id,split,text,region,Tags
9018,77,valid,Fói’a oficiale din Belgradu publica in nrulu s...,Transylvania,"[0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, ..."
9019,926,valid,Mai multu se intereséza lumea de cele palide ...,Transylvania,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9020,8366,valid,Doar pentru atât păcat ar fi fost de râul de ...,Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9021,5990,valid,) România perde dar o provincie .,Wallachia,"[0, 0, 0, 0, 0, 0, 0]"
9022,8452,valid,Uneori un cuvînt aspru sau de ocară ne aduce o...,Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [4]:
df_train = df_train.drop(columns='split')

df_train.tail()

,id,text,region,Tags
9018,77,Fói’a oficiale din Belgradu publica in nrulu s...,Transylvania,"[0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, ..."
9019,926,Mai multu se intereséza lumea de cele palide ...,Transylvania,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9020,8366,Doar pentru atât păcat ar fi fost de râul de ...,Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9021,5990,) România perde dar o provincie .,Wallachia,"[0, 0, 0, 0, 0, 0, 0]"
9022,8452,Uneori un cuvînt aspru sau de ocară ne aduce o...,Bessarabia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [5]:
df_train.text[1]

'1 ) Asupra cestiunii , avem în romaneste oarecare noţiuni în doi lucrări : a ) Critica apriorismului si empirismului , de Petru P. NeguJescu , p. 125 sqq .'

In [7]:
#subtask1
features = 'text'
target = 'region'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(y)

X

<9023x247060 sparse matrix of type '<class 'numpy.float64'>'
	with 3714940 stored elements in Compressed Sparse Row format>

In [8]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=2, scoring='f1_macro')

array([0.81576338, 0.81595631])

In [9]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [10]:
y_preds = enc.inverse_transform(y_preds)

In [11]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.id,
    "answer": y_preds
})

df_1.head()

,subtaskID,datapointID,answer
0,1,4319,Bessarabia
1,1,8346,Wallachia
2,1,3963,Moldavia
3,1,3151,Moldavia
4,1,3272,Moldavia


In [12]:
#subtask2 AAHahsHSDAHDASHD
# voi face un dataset nou cu tokenuri si taguri pentru fiecare token si dupa voi antrena modelul pe acela
df_train.head()

,id,text,region,Tags
0,969,Aci o asceptá o duzina de juni-junisiroi tomna...,Transylvania,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2992,"1 ) Asupra cestiunii , avem în romaneste oarec...",Moldavia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,3926,Fe­ nomenul e acelaşi în toate emoţiunile pute...,Moldavia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,8619,"al Bucovineĭ , substratul , formarea , desvolt...",Bessarabia,"[0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,3670,Este interesant a vedea luptele şi silinţele c...,Moldavia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [13]:
def get_tag_list(val):
    val = str(val).replace(r'[', '').replace(r']', '')
    val = val.split(', ')
    val = [int(val) for val in val]
    return val

get_tag_list(df_train.Tags[0])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [14]:
words = []
y = []

for _, row in good_train.iterrows():
    tokens = row.text.split()
    labels = get_tag_list(row.Tags)
    if len(tokens) == len(labels):
        words.extend(tokens)
        y.extend(labels)

print(words[:3], y[:3])

['Aci', 'o', 'asceptá'] [0, 0, 0]


In [48]:
import numpy as np
X = []

def get_features(word):
    features = {
        "word": word,
        "lower": word.lower(),
        "is_upper": word.isupper(),
        "is_title": word.istitle(),
        "suffix3": word[-3:],
        "prefix3": word[:3],
    }
    return features

for word in words:
    X.append(get_features(word))

X = np.array(X)

X[0]

{'word': 'Aci',
 'lower': 'aci',
 'is_upper': False,
 'is_title': True,
 'suffix3': 'Aci',
 'prefix3': 'Aci'}

In [49]:
words_final = []

for _, row in df_test.iterrows():
    tokens = row.text.split()
    words_final.extend(tokens)

X_final = []

for word in words_final:
    X_final.append(get_features(word))

X_final = np.array(X_final)
X_final[0]

{'word': 'Vă',
 'lower': 'vă',
 'is_upper': False,
 'is_title': True,
 'suffix3': 'Vă',
 'prefix3': 'Vă'}

In [50]:
X = X[:50000]
y = y[:50000]

In [51]:
from sklearn.feature_extraction import DictVectorizer
vect = DictVectorizer()
X = vect.fit_transform(X)

from sklearn.decomposition import TruncatedSVD
decom = TruncatedSVD(n_components=300)
X = decom.fit_transform(X)

X

array([[-7.11871522e-15,  6.24487278e-03,  3.88009038e-15, ...,
        -8.90181404e-03,  3.03527195e-02, -4.69342970e-02],
       [-1.78059351e-16,  2.76262575e-05, -3.63901911e-15, ...,
        -2.61643604e-03, -1.35900097e-03,  6.54431357e-05],
       [-3.47479900e-17,  9.55101993e-09,  1.65734837e-15, ...,
        -2.19799749e-04,  4.01264431e-04, -2.41041060e-03],
       ...,
       [ 1.11995895e-16,  1.05400132e-06,  7.24840890e-15, ...,
         1.78677542e-02, -4.06753440e-03,  1.17390462e-02],
       [-6.00112097e-17,  9.27543880e-07, -2.74635202e-14, ...,
        -1.22594231e-03, -1.11666712e-03, -1.25931083e-03],
       [ 2.00000000e+00, -2.30579558e-14, -1.56278853e-15, ...,
         4.01945906e-16,  8.60113801e-16, -7.59795840e-16]],
      shape=(50000, 300))

In [64]:
model = LogisticRegression()

cross_val_score(model, X, y, cv=2, scoring='f1_macro')

array([0.12739404, 0.12399693])

In [65]:
model.fit(X, y)

LogisticRegression()

In [77]:
from tqdm import tqdm
y_preds = []

for _, row in tqdm(df_test.iterrows()):
    sent = row.text
    words = sent.split()
    feats = [get_features(word) for word in words]
    feats = vect.transform(feats)
    feats = decom.transform(feats)
    preds = model.predict(feats)
    y_preds.append(preds)

y_preds[:5]

1003it [00:27, 36.50it/s]


[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])]

In [78]:
def get_string(preds):
    preds = [str(pred) for pred in list(preds)]
    string = ', '.join(preds)
    string = ''.join([r'[', string])
    string = ''.join([string, r']'])
    return string

real_preds = [get_string(pred) for pred in y_preds]
real_preds[:5]

['[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]',
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]',
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]',
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]',
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]']

In [79]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": real_preds
})

df_2.head()

,subtaskID,datapointID,answer
0,2,4319,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,2,8346,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,2,3963,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,2,3151,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,2,3272,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [80]:
submission = pd.concat([df_1, df_2])
submission

,subtaskID,datapointID,answer
0,1,4319,Bessarabia
1,1,8346,Wallachia
2,1,3963,Moldavia
3,1,3151,Moldavia
4,1,3272,Moldavia
...,...,...,...
998,2,108,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
999,2,9642,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1000,2,3731,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1001,2,3891,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [81]:
submission.to_csv("submission.csv", index=False)